# 03. Preprocessing Data

Mempelajari cara imputasi, standardisasi, dan One-Hot Encoding bekerja pada fitur data latih.

**Prasyarat:** Tahap 01 selesai.

**Keluaran:** `preview_preprocessing_train.csv` sebagai bahan pembelajaran.

Pilih kernel Python 3.12 dengan dependency proyek, lalu **Restart Kernel → Run All**. Tahap 01–05 dijalankan secara berurutan. Proyek edukasi data science, bukan diagnosis medis klinis.


## A. Inisialisasi Lingkungan dan Lokasi Proyek
Mempersiapkan konstanta fitur dan folder kerja.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
from datetime import datetime, timezone
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Dataset/Data_Asli/nhanes_depression_raw.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari dalam folder proyek lengkap dengan Dataset.')
TARGET = "Depression"
LABELS = ["Not Depressed", "Depressed"]
DISPLAY = {
    "Not Depressed": "Tidak Terindikasi Depresi Mayor",
    "Depressed": "Berisiko Mengalami Depresi (PHQ-9 ≥ 10)"
}

CATEGORICAL = [
    "Gender", "Education Level", "Marital Status", "Household Income",
    "Trouble Sleeping History", "Vigorous Recreation", "Moderate Recreation",
    "Current Smoker", "Memory Problems", "Cant Work", "Limited Work"
]
NUMERIC = [
    "Age", "Sleep Hours", "Sedentary Time", "BMI", "Systolic", "Diastolic", "Pulse"
]
FEATURES = CATEGORICAL + NUMERIC

RANGES = {
    "Age": (18, 120),
    "Sleep Hours": (1.0, 24.0),
    "Sedentary Time": (0.0, 1440.0),
    "BMI": (10.0, 90.0),
    "Systolic": (60.0, 260.0),
    "Diastolic": (30.0, 160.0),
    "Pulse": (30.0, 220.0),
}

SEED = 42
REPORTS = ROOT / "Source_Code/reports"
EDA = REPORTS / "eda"
MODELING = REPORTS / "modeling"
ANALYSIS = REPORTS / "analysis"
PROCESSED = ROOT / "Dataset/Data_Olahan"
MODELS = ROOT / "Model"


for folder in [EDA, MODELING, ANALYSIS, PROCESSED, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)
source = ROOT / 'Dataset/Data_Asli/nhanes_depression_raw.csv'
def save_json(path, value):
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False,
                   default=lambda x: x.item() if isinstance(x, np.generic) else str(x)),
        encoding="utf-8"
    )

def save_figure(path):
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    display(Image(filename=str(path)))
    plt.close()

print('Proyek:', ROOT)


Proyek: D:\KULIAH\sem-5\MACHINE LEARNING\PJBL\prediksi_gangguan_depresi


## B. Memuat Data Latih
Data uji tidak disentuh dalam proses fitting transformer.


In [2]:
split_file = PROCESSED / 'notebook_split.json'
if not split_file.exists():
    raise FileNotFoundError('Jalankan 01_persiapan_data.ipynb terlebih dahulu.')
split_state = json.loads(split_file.read_text(encoding='utf-8'))
for relative, expected in split_state['input_hashes'].items():
    if hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() != expected:
        raise ValueError('Data berubah. Jalankan ulang notebook 01: ' + relative)
raw = pd.read_csv(source, low_memory=False)
saved_clean = pd.read_csv(PROCESSED / 'cleaned.csv', low_memory=False)
clean = saved_clean[FEATURES + [TARGET]].copy()
ids = saved_clean['source_seqn']
profile_hashes = saved_clean['profile_hash'] if 'profile_hash' in saved_clean.columns else pd.util.hash_pandas_object(clean[FEATURES], index=False).astype(str)
conflict_hashes = set()
duplicate_mask = pd.Series([False] * len(raw))
X, y = clean[FEATURES], clean[TARGET]
train_idx, test_idx = split_state['train_indices'], split_state['test_indices']
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print('Data Latih:', X_train.shape, '| Data Uji:', X_test.shape)


Data Latih: (29007, 18) | Data Uji: (7252, 18)


## C. Membangun Pipeline Preprocessing
Pipeline menggabungkan SimpleImputer, StandardScaler, dan OneHotEncoder.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

def make_preprocessor():
    """Membangun objek ColumnTransformer baru yang belum di-fit."""
    num_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    return ColumnTransformer([
        ("numeric", num_pipeline, NUMERIC),
        ("categorical", cat_pipeline, CATEGORICAL)
    ])


## D. Fit dan Transform pada Data Latih (Preview Pembelajaran)
Menghasilkan matriks fitur ter-encode untuk inspeksi.


In [4]:
preview_preprocessor = make_preprocessor()
preview_values = preview_preprocessor.fit_transform(X_train)
preview_cols = preview_preprocessor.get_feature_names_out()
preview = pd.DataFrame(preview_values, columns=preview_cols, index=X_train.index)
print('Fitur sebelum encoding:', X_train.shape[1], '| Setelah encoding:', preview.shape[1])
display(X_train.head(), preview.head())
preview.to_csv(PROCESSED / 'preview_preprocessing_train.csv', index_label='clean_index')
print('Tahap 03 Selesai. Lanjut ke 04_training_model.ipynb')


Fitur sebelum encoding: 18 | Setelah encoding: 53


,Gender,Education Level,Marital Status,Household Income,Trouble Sleeping History,Vigorous Recreation,Moderate Recreation,Current Smoker,Memory Problems,Cant Work,Limited Work,Age,Sleep Hours,Sedentary Time,BMI,Systolic,Diastolic,Pulse
23551,Male,College Graduate or Above,Married,Below $25K,No,No,No,Never,No,No,No,52,8.0,180.0,25.80,140.0,96.0,68.0
11111,Male,Less Than 9th Grade,Widowed,Missing,Yes,No,No,Some Days,Yes,Yes,Yes,53,5.0,30.0,23.35,126.0,86.0,60.0
2786,Female,9-11th Grade,Widowed,Below $15K,No,No,No,Missing,No,No,No,80,9.0,300.0,40.86,150.0,70.0,62.0
32499,Female,9-11th Grade,Married,Below $20K,No,No,No,Missing,No,No,No,49,8.0,600.0,31.60,216.0,98.0,52.0
19457,Female,High School,Married,Below $10K,No,No,No,Never,No,No,No,44,4.0,180.0,24.10,142.0,92.0,66.0


,numeric__Age,numeric__Sleep Hours,numeric__Sedentary Time,numeric__BMI,numeric__Systolic,numeric__Diastolic,numeric__Pulse,categorical__Gender_Female,categorical__Gender_Male,categorical__Education Level_9-11th Grade,...,categorical__Current Smoker_Every Day,categorical__Current Smoker_Missing,categorical__Current Smoker_Never,categorical__Current Smoker_Some Days,categorical__Memory Problems_No,categorical__Memory Problems_Yes,categorical__Cant Work_No,categorical__Cant Work_Yes,categorical__Limited Work_No,categorical__Limited Work_Yes
23551,0.225098,0.586180,-0.748064,-0.389168,0.675952,1.418560,-0.215458,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
11111,0.278382,-1.328249,-1.495760,-0.712880,0.290229,0.960485,-0.727473,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
2786,1.717058,1.224323,-0.149907,1.600669,0.951468,0.227564,-0.599469,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
32499,0.065245,0.586180,1.345486,0.377171,2.769876,1.510175,-1.239488,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
19457,-0.201177,-1.966392,-0.748064,-0.613784,0.731055,1.235330,-0.343462,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0


Tahap 03 Selesai. Lanjut ke 04_training_model.ipynb
